In [ ]:
from torch.utils.data import Dataset
from typing import Callable, Any, Tuple, Dict
import torchvision
from torch.nn import functional as th_f
from torchvision.transforms import Compose, Resize
import torch as th
from os.path import join, isdir, exists, splitext, basename
import pandas as pd
import glob

In [ ]:
from marl_classification.data import KineticsDataset

In [ ]:
dataset = KineticsDataset("/run/media/samuel/2682F1C682F19B0F", lambda a: a)

In [ ]:
data, label = dataset[210]

In [ ]:
data.size(), label

In [ ]:
len(dataset.class_to_idx)

In [ ]:
resize = Resize((640, 301))

In [ ]:
data_2 = resize(data)

In [ ]:
data.size(), data_2.size()

In [ ]:
len(dataset)

In [ ]:
class ResizeFrame:
    def __init__(self, width: int, height: int) -> None:
        self.__width = width
        self.__height = height

    def __call__(self, img_data: th.Tensor) -> th.Tensor:
        frames = img_data.size(-1)
        img_data = th_f.interpolate(img_data.unsqueeze(0), size=(self.__width, self.__height, frames), mode="trilinear")
        img_data = img_data.squeeze(0)
        return img_data

In [ ]:
resize_custom = ResizeFrame(360, 640)

In [ ]:
data_3 = resize_custom(data.to(th.float))

In [ ]:
data_3.size()

In [ ]:
class KineticsDataset(Dataset):
    def __init__(
        self, res_path: str, img_transform: Callable[[Any], th.Tensor]
    ) -> None:
        kinetics_dataset_path = join(
            res_path, "downloaded", "kinetics700_2020"
        )

        assert exists(kinetics_dataset_path) and isdir(kinetics_dataset_path)

        self.__videos_path = join(kinetics_dataset_path, "videos")

        train_df = pd.read_csv(join(kinetics_dataset_path, "train.csv"))
        video_ids = set(train_df["youtube_id"].tolist())

        self.__all_videos = [
            splitext(basename(f))[0]
            for f in glob.glob(join(self.__videos_path, "*.mp4"))
            if splitext(basename(f))[0] in video_ids
        ]

        tmp_all_video = set(self.__all_videos)
        self.__all_labels = {
            row["youtube_id"]: row["label"]
            for _, row in train_df.iterrows()
            if row["youtube_id"] in tmp_all_video
        }

        self.__class_to_idx = {
            label: i for i, label in enumerate(train_df["label"].unique())
        }

        self.__transform =

        self.__img_transform = img_transform



    def __getitem__(self, index: int) -> Tuple[th.Tensor, th.Tensor]:
        video_path = self.__all_videos[index]
        video_label = self.__all_labels[video_path]

        video = torchvision.io.VideoReader(
            join(self.__videos_path, video_path + ".mp4"), "video"
        )

        video.set_current_stream("video")

        video_data = th.stack([frame["data"] for frame in video], dim=-1)

        return self.__img_transform(video_data), th.tensor(
            self.class_to_idx[video_label]
        )

    @property
    def class_to_idx(self) -> Dict[str, int]:
        return self.__class_to_idx

    def __len__(self) -> int:
        return len(self.__all_videos)